In [1]:
# !pip install gensim

In [2]:
# !pip install tabulate

In [3]:
import pandas as pd
import re
import spacy
import numpy as np
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tabulate import tabulate  # for displaying results in a table

# Load SpaCy for tokenization
nlp = spacy.load('en_core_web_sm')

C:\Users\batti\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Preprocessing function
def preprocess_text(text, lemmatize_words=True, remove_stop_words=True):
    text = re.sub(r'\s+', ' ', text).strip()
    doc = nlp(text)
    
    if lemmatize_words:
        tokens = [token.lemma_ for token in doc]
    else:
        tokens = [token.text for token in doc]
    
    if remove_stop_words:
        tokens = [token for token in tokens if not nlp.vocab[token].is_stop]
    
    # Remove symbols (punctuation and other non-word characters)
    tokens = [re.sub(r'[^\w\s]', '', token) for token in tokens]
    
    # Remove empty tokens
    tokens = [token for token in tokens if token]
    
    return tokens

def compute_tfidf_similarity(original_docs, query_docs):
    # Combining both train and test dataset.
    all_documents = pd.concat([original_docs['processed_document'], query_docs['processed_document']])
    
    # Creating the vectorizer and fitting the TF-IDF model
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(all_documents)
    
    # Get the feature names (words) extracted by the vectorizer
    feature_names = vectorizer.get_feature_names_out()
    
    print(f"----------------TF-IDF--------------")
    # Print the number of unique words (features) in the vocabulary
    print(f"Number of unique words: {len(feature_names)}")
    
    # Print the vector size for each word
    print(f"Size of TF-IDF matrix: {tfidf_matrix.shape}")
    
    # Converting the matrix to dense format to see specific vectors
    dense_tfidf_matrix = tfidf_matrix.todense()
    
    # Printing the TF-IDF vector for the first document as an example
    first_doc_vector = np.array(dense_tfidf_matrix[0])
    print(f"TF-IDF vector for the first document:\n{first_doc_vector}")
    
    # Splitting the matrix back into original and query tf-idf matrices
    original_tfidf = tfidf_matrix[:len(original_docs)]
    query_tfidf = tfidf_matrix[len(original_docs):]
    
    # Returning cosine similarity between the query and original documents
    return cosine_similarity(query_tfidf, original_tfidf)

# Function to create Word2Vec embeddings and similarities
def compute_word2vec_similarity(original_docs, query_docs):
    # Combine the train and test dataset
    all_documents = original_docs['tokenized_document'].tolist() + query_docs['tokenized_document'].tolist()
    # Creating embedding for each word
    word2vec_model = Word2Vec(sentences=all_documents, vector_size=100, window=2, min_count=1, sg=1, epochs=15)
    # print(f"----------------Word2Vec--------------")
    # Calculating the document embedding by averaging the all the word embedding of that document
    original_docs_embeddings = np.array([create_doc_embedding(word2vec_model, doc) for doc in original_docs['tokenized_document']])
    query_docs_embeddings = np.array([create_doc_embedding(word2vec_model, doc) for doc in query_docs['tokenized_document']])
    return cosine_similarity(query_docs_embeddings, original_docs_embeddings)

# # Uncomment the below to run for different set of params
# # Function to create Word2Vec embeddings and similarities
# def compute_word2vec_similarity(original_docs, query_docs):
#     # Combine the train and test dataset
#     all_documents = original_docs['tokenized_document'].tolist() + query_docs['tokenized_document'].tolist()
#     # Creating embedding for each word
#     word2vec_model = Word2Vec(sentences=all_documents, vector_size=100, window=1, min_count=10, sg=1, epochs=10)
#     print(f"----------------Word2Vec--------------")
#     # Calculating the document embedding by averaging the all the word embedding of that document
#     original_docs_embeddings = np.array([create_doc_embedding(word2vec_model, doc) for doc in original_docs['tokenized_document']])
#     query_docs_embeddings = np.array([create_doc_embedding(word2vec_model, doc) for doc in query_docs['tokenized_document']])
#     return cosine_similarity(query_docs_embeddings, original_docs_embeddings)

# Function to create document embeddings from Word2Vec
def create_doc_embedding(model, document):
    valid_words = [word for word in document if word in model.wv]
    if valid_words:
        return np.mean([model.wv[word] for word in valid_words], axis=0)
    else:
        return np.zeros(model.vector_size)

# Function to print the results in a single table for each query
def print_similarity_tables(tfidf_similarities, word2vec_similarities, query_docs):
    for i, query_doc in enumerate(query_docs['documents']):
        tfidf_sim = tfidf_similarities[i]
        tfidf_top_5_indices = tfidf_sim.argsort()[::-1][:5]
        
        tfidf_results = [f"document {idx + 1}" for idx in tfidf_top_5_indices]
        tfidf_similarities_scores = [round(tfidf_sim[idx], 4) for idx in tfidf_top_5_indices]
        
        w2v_sim = word2vec_similarities[i]
        w2v_top_5_indices = w2v_sim.argsort()[::-1][:5]
        
        w2v_results = [f"document {idx + 1}" for idx in w2v_top_5_indices]
        w2v_similarities_scores = [round(w2v_sim[idx], 4) for idx in w2v_top_5_indices]
        
        # Prepare the table data for the current query
        table_data = []
        # table_data.append(["Query Document", f"Query {i + 1}: {query_doc}"])
        table_data.append(["TF-IDF Documents", "TF-IDF Similarity", "Word2Vec Documents", "Word2Vec Similarity"])
        
        for tfidf_doc, tfidf_score, w2v_doc, w2v_score in zip(tfidf_results, tfidf_similarities_scores, w2v_results, w2v_similarities_scores):
            table_data.append([tfidf_doc, tfidf_score, w2v_doc, w2v_score])
        
        # Print the table using tabulate for the current query
        print(f"\nResults for Query {i + 1}: {query_doc}\n")
        try:
            print(tabulate(table_data, headers="firstrow", tablefmt="fancy_grid"))
        except Exception as e:
            print("Tabulate failed, falling back to manual table printing.")
            for row in table_data:
                print(f"{row[0]}: {row[1]}")
            print("-" * 50)

In [ ]:
# Main function to process and compute similarities
def main():
    # Define preprocessing options
    lemmatize_words = False
    remove_stop_words = False
    
    # Step 1: Read the original 10 documents from the CSV file
    df_original = pd.read_csv('data\data.csv')
    
    # Step 2: Define the query documents
    df_test = pd.DataFrame([
        "Artificial intelligence is set to take over most jobs in near future.",
        "The use of artificial intelligence in healthcare industry is more and more every day.",
        "The use of AI in healthcare industry is more and more every day.",
        "The use of AI in medical care is more and more every day."
    ], columns=['documents'])
    
    # Step 3: Preprocess both original and query documents with specified parameters
    df_original['tokenized_document'] = df_original['documents'].apply(lambda x: preprocess_text(x, lemmatize_words, remove_stop_words))
    df_test['tokenized_document'] = df_test['documents'].apply(lambda x: preprocess_text(x, lemmatize_words, remove_stop_words))
    
    df_original['processed_document'] = df_original['tokenized_document'].apply(lambda x: ' '.join(x))
    df_test['processed_document'] = df_test['tokenized_document'].apply(lambda x: ' '.join(x))
    
    # Step 4: Compute TF-IDF and Word2Vec similarities
    tfidf_similarities = compute_tfidf_similarity(df_original, df_test)
    
    word2vec_similarities = compute_word2vec_similarity(df_original, df_test)
    
    # Step 5: Print similarities side by side in a table
    print_similarity_tables(tfidf_similarities, word2vec_similarities, df_test)

# Run the main function
if __name__ == "__main__":
    main()

----------------TF-IDF--------------
Number of unique words: 331
Size of TF-IDF matrix: (14, 331)
TF-IDF vector for the first document:
[[0.         0.         0.         0.         0.         0.10809727
  0.         0.         0.         0.         0.15966289 0.10809727
  0.1403691  0.         0.         0.         0.         0.
  0.         0.15966289 0.08204245 0.1403691  0.06543623 0.
  0.         0.10809727 0.         0.1403691  0.         0.
  0.         0.         0.         0.         0.         0.
  0.1403691  0.         0.         0.         0.1403691  0.
  0.17843892 0.08204245 0.24298258 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.1403691  0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.12149129 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.12149129 0.1403691  0.         0.         0.         0.
  0.    